# LCEL and OpenAI and Prompt Template with History

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableLambda

llm = ChatOpenAI(
    model="qwen/qwen3-vl-235b-a22b-thinking",
    openai_api_key="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",
    openai_api_base="https://openrouter.ai/api/v1",
    extra_body={
        "reasoning": {"enabled": True}
    },
    temperature=0
)

analysis_prompt = ChatPromptTemplate.from_messages(
    [
      ("system", "You are a smart analysis agent. Study the history and the user prompt and come with a step by step analysis"),
      MessagesPlaceholder(variable_name="chat_history"),
      ("human", "{user_input}")
    ]
  )

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You have the analysis and chat history. Now come up with the answer."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "Analysis {analysis}")
    ]
)

chat_history = []

def extract_analysis(msg):
    return {"analysis": msg.content,"chat_history":chat_history}

analysis_step = RunnableLambda(extract_analysis)

chain = analysis_prompt | llm | analysis_step | final_prompt | llm

def chat(user_input: str) -> str:
    global chat_history

    answer = chain.invoke({
        "chat_history": chat_history,
        "user_input": user_input
    })

    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=str(answer)))

    return answer.content


if __name__ == "__main__":
    print("🔹 LangChain OpenAI (non-chat) + OpenRouter")
    print("Type 'exit' to quit\n")

    while True:
        user_input = input("User: ")

        if user_input.lower() in {"exit", "quit"}:
            print("Goodbye 👋")
            break

        reply = chat(user_input)
        print("Assistant:", reply)
        print()




# RAG

In [ ]:
import os
from typing import List

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableLambda
from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


# =========================================================
# 1️⃣ LLM CONFIG (same style as your original app)
# =========================================================

llm = ChatOpenAI(
    model="qwen/qwen3-vl-235b-a22b-thinking",
    openai_api_key="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",
    openai_api_base="https://openrouter.ai/api/v1",
    extra_body={"reasoning": {"enabled": True}},
    temperature=0
)


# =========================================================
# 2️⃣ KNOWLEDGE BASE (replace with files later)
# =========================================================

raw_docs: List[Document] = [
    Document(page_content="""
LangChain Expression Language (LCEL) allows developers to compose
AI pipelines using the pipe (|) operator. It is deterministic and debuggable.
"""),
    Document(page_content="""
Retrieval-Augmented Generation (RAG) improves LLM accuracy by retrieving
relevant chunks of information from external knowledge sources.
"""),
    Document(page_content="""
FAISS is a vector similarity search library used to build fast
in-memory vector databases for retrieval.
""")
]


# =========================================================
# 3️⃣ CHUNKING (MANDATORY FOR RAG)
# =========================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks: List[Document] = text_splitter.split_documents(raw_docs)


# =========================================================
# 4️⃣ EMBEDDINGS (NON-OPENAI, NON-DEPRECATED)
# =========================================================

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# =========================================================
# 5️⃣ VECTOR STORE + RETRIEVER
# =========================================================

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


# =========================================================
# 6️⃣ RETRIEVAL RUNNABLE (LCEL STYLE)
# =========================================================

def retrieve_context(inputs):
    query = inputs["user_input"]
    chat_history = inputs["chat_history"]

    retrieved_docs = retriever.invoke(query)

    context = "\n\n".join(
        f"[Chunk {i}]\n{doc.page_content}"
        for i, doc in enumerate(retrieved_docs)
    )

    return {
        "context": context,
        "user_input": query,
        "chat_history": chat_history
    }

retrieval_step = RunnableLambda(retrieve_context)


# =========================================================
# 7️⃣ ANALYSIS PROMPT
# =========================================================

analysis_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a smart assistant.\n"
            "Use the retrieved context when it is relevant.\n"
            "If the question is conversational or based on chat history "
            "(e.g., user's name, greetings, clarifications), answer normally.\n"
            "Do NOT hallucinate external factual information.\n\n"
            "Context (if any):\n{context}"
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{user_input}")
    ]
)

# =========================================================
# 8️⃣ ANALYSIS EXTRACTION
# =========================================================
chat_history: List = []
def extract_analysis(msg):
    return {"analysis": msg.content, "chat_history":chat_history}

analysis_step = RunnableLambda(extract_analysis)


# =========================================================
# 9️⃣ FINAL PROMPT
# =========================================================

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You have the analysis. Produce a clear final answer."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{analysis}")
    ]
)


# =========================================================
# 🔗 10️⃣ FULL LCEL RAG CHAIN
# =========================================================

chain = (
    retrieval_step
    | analysis_prompt
    | llm
    | analysis_step
    | final_prompt
    | llm
)


# =========================================================
# 11️⃣ CHAT LOOP (HISTORY PRESERVED)
# =========================================================



def chat(user_input: str) -> str:
    global chat_history

    answer = chain.invoke({
        "user_input": user_input,
        "chat_history": chat_history
    })

    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=answer.content))

    return answer.content


# =========================================================
# 12️⃣ CLI ENTRYPOINT
# =========================================================

if __name__ == "__main__":
    print("🔹 LCEL RAG Application (Stable & Correct)")
    print("Type 'exit' to quit\n")

    while True:
        user_input = input("User: ")

        if user_input.lower() in {"exit", "quit"}:
            print("Goodbye 👋")
            break

        reply = chat(user_input)
        print("\nAssistant:", reply)
        print()

# Agent = RAG Agent + Resources + Tools

In [ ]:
# =========================================================
# 1️⃣ IMPORTS
# =========================================================
import requests
from typing import List

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.tools.retriever import create_retriever_tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
# =========================================================
# 2️⃣ STATIC RESOURCES
# =========================================================
RESOURCES = """
Company Policies:
- Office hours are 9 AM to 6 PM IST.
- Remote work is allowed on Fridays.

Product Information:
- EduMate is a personalized learning platform.
- EduMate connects students with expert mentors.
"""

# =========================================================
# 3️⃣ LLM
# =========================================================
llm = ChatOpenAI(
    model="qwen/qwen3-vl-235b-a22b-thinking",
    openai_api_key="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",
    openai_api_base="https://openrouter.ai/api/v1",
    extra_body={"reasoning": {"enabled": True}},
    temperature=0
)

# =========================================================
# 4️⃣ RAG PIPELINE
# =========================================================
raw_docs: List[Document] = [
    Document(page_content="EduMate supports personalized learning paths."),
    Document(page_content="EduMate uses AI-based mentor matching."),
    Document(page_content="EduMate tracks student progress."),
    Document(page_content="Liquibase is used for database schema migration."),
    Document(page_content="Spring Boot is used for backend services."),
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(raw_docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_tool = create_retriever_tool(
    retriever=retriever,
    name="knowledge_base_search",
    description="Search internal EduMate knowledge base"
)

# =========================================================
# 5️⃣ WEATHER TOOL
# =========================================================
@tool
def get_weather(city: str) -> str:
    """Get real-time weather for a city."""
    api_key = "056b5b3fc2828a9d2932a4e088bf5041"
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"
    response = requests.get(url, timeout=10)
    if response.status_code != 200:
        return f"Failed to fetch weather for {city}"
    data = response.json()
    return f"Weather in {city}: {data['weather'][0]['description']}, {data['main']['temp']}°C"

# =========================================================
# 6️⃣ MULTIPLICATION TOOL
# =========================================================
@tool
def multiply(numbers: str) -> str:
    """Multiply two numbers. Input format: a,b"""
    a, b = map(int, numbers.split(","))
    return str(a * b)

# =========================================================
# 7️⃣ CREATE AGENT (NEW WAY)
# =========================================================
agent = create_agent(
    llm,
    tools=[rag_tool, get_weather, multiply],
    system_prompt=f"You are a helpful AI agent.\n\nRESOURCES:\n{RESOURCES}"
)

# =========================================================
# 8️⃣ RUN EXAMPLES
# =========================================================
if __name__ == "__main__":

    print("\n--- RAG QUERY ---")
    result = agent.invoke({"messages": [{"role": "user", "content": "How does EduMate work?"}]})
    print(result["messages"][-1].content)

    print("\n--- WEATHER QUERY ---")
    result = agent.invoke({"messages": [{"role": "user", "content": "What is the weather in Delhi?"}]})
    print(result["messages"][-1].content)

    print("\n--- MULTIPLICATION QUERY ---")
    result = agent.invoke({"messages": [{"role": "user", "content": "Multiply 12 and 9"}]})
    print(result["messages"][-1].content)

    print("\n--- RESOURCES + RAG QUERY ---")
    result = agent.invoke({"messages": [{"role": "user", "content": "Tell me about EduMate and office hours"}]})
    print(result["messages"][-1].content)

# Multi Agent - Main Agent + Sub agents as Tools

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent

# Create a subagent
subagent = create_agent(model="anthropic:claude-sonnet-4-20250514", tools=[...])

# Wrap it as a tool
@tool("research", description="Research a topic and return findings")
def call_research_agent(query: str):
    result = subagent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

# Main agent with subagent as a tool
main_agent = create_agent(model="anthropic:claude-sonnet-4-20250514", tools=[call_research_agent])

# MCP - Tryout (Not Tested)

## MCP Server (mcp_server.py)

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("dynamic-mcp")

# -------------------------
# Tool
# -------------------------
@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

# -------------------------
# Resource
# -------------------------
@mcp.resource("info://about")
def about():
    return "This MCP server provides a simple math tool."

if __name__ == "__main__":
    mcp.run()

## MCP Client - Connects to MCP Server + Converts MCP Tools to Langchain Tools (mcp_tool_loader.py)

In [ ]:
import asyncio
from mcp.client.stdio import stdio_client
from mcp import ClientSession
from langchain.tools import Tool

class MCPToolLoader:
    def __init__(self):
        self.session = None

    async def connect(self):
        self.ctx = stdio_client("python", ["mcp_server.py"])
        self.read, self.write = await self.ctx.__aenter__()
        self.session = ClientSession(self.read, self.write)
        await self.session.initialize()

    def load_tools(self):
        """
        Fetch MCP tools and convert them into LangChain Tool objects
        """
        mcp_tools = asyncio.run(self.session.list_tools())
        langchain_tools = []

        for tool_meta in mcp_tools:
            name = tool_meta["name"]
            description = tool_meta.get("description", "")

            def make_func(tool_name):
                def _call(**kwargs):
                    return asyncio.run(
                        self.session.call_tool(tool_name, kwargs)
                    )
                return _call

            langchain_tools.append(
                Tool(
                    name=name,
                    description=description,
                    func=make_func(name)
                )
            )

        return langchain_tools

## Actual Agent with Tools from MCP Client (agent.py) Uses **create_agent()**

In [ ]:
# =========================================================
# IMPORTS
# =========================================================
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

from mcp_tool_loader import MCPToolLoader

# =========================================================
# LLM
# =========================================================
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# =========================================================
# MCP TOOL LOADING (BEFORE create_agent)
# =========================================================
loader = MCPToolLoader()
import asyncio
asyncio.run(loader.connect())

tools = loader.load_tools()

# =========================================================
# CREATE AGENT (YOUR REQUIRED FORMAT)
# =========================================================
agent = create_agent(
    llm=llm,
    tools=tools,
    system_prompt="You are a helpful autonomous agent. Use tools when needed."
)

# =========================================================
# RUN
# =========================================================
if __name__ == "__main__":
    result = agent.invoke({
        "messages": [
            {"role": "user", "content": "What is 8 + 12?"}
        ]
    })

    print("\n🤖 Agent Answer:")
    print(result["messages"][-1].content)